# Document Summarizer

Paste any text (e.g. a Wikipedia article) and get back a structured summary:
- A one-paragraph summary
- Three key points
- A sentiment (positive / neutral / negative)

Claude is instructed to respond using XML tags, which we parse and print cleanly (the raw API response is never shown to the user).

In [1]:
%pip install -q anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import re
from textwrap import dedent
from dotenv import load_dotenv
from anthropic import Anthropic
import anthropic
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
model = "claude-sonnet-4-6"


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 3000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
SYSTEM_PROMPT = dedent("""
    You are a precise document summarizer. Given input text, respond with a structured
    summary and nothing else — no preamble, no closing remarks, no text outside the tags
    below.

    Respond using exactly this format:

    <summary>
    A single paragraph (3-5 sentences) summarizing the text.
    </summary>

    <key_points>
    <point>First key point.</point>
    <point>Second key point.</point>
    <point>Third key point.</point>
    </key_points>

    <sentiment>positive|neutral|negative</sentiment>

    Rules:
    - <key_points> must contain exactly three <point> elements.
    - <sentiment> must be exactly one word: positive, neutral, or negative.
    - Do not wrap the output in markdown code fences.
    - Do not include any text before <summary> or after </sentiment>.
""").strip()

In [4]:
def get_structured_summary(text):
    messages = []
    add_user_message(messages, text)
    return chat(messages, system=SYSTEM_PROMPT, temperature=0)

In [5]:
VALID_SENTIMENTS = {"positive", "neutral", "negative"}


def parse_summary(raw):
    summary_match = re.search(r"<summary>(.*?)</summary>", raw, re.DOTALL)
    points_match = re.search(r"<key_points>(.*?)</key_points>", raw, re.DOTALL)
    sentiment_match = re.search(r"<sentiment>(.*?)</sentiment>", raw, re.DOTALL)

    summary = summary_match.group(1).strip() if summary_match else None

    key_points = []
    if points_match:
        key_points = [
            p.strip() for p in re.findall(r"<point>(.*?)</point>", points_match.group(1), re.DOTALL)
        ]

    sentiment = sentiment_match.group(1).strip().lower() if sentiment_match else None
    if sentiment not in VALID_SENTIMENTS:
        sentiment = sentiment or "unknown"

    return {"summary": summary, "key_points": key_points, "sentiment": sentiment}

In [6]:
SENTIMENT_EMOJI = {"positive": "🟢", "neutral": "⚪", "negative": "🔴"}


def print_summary(parsed):
    print("SUMMARY")
    print("-" * 60)
    print(parsed["summary"] or "(no summary returned)")

    print("\nKEY POINTS")
    print("-" * 60)
    if parsed["key_points"]:
        for i, point in enumerate(parsed["key_points"], 1):
            print(f"{i}. {point}")
    else:
        print("(no key points returned)")

    print("\nSENTIMENT")
    print("-" * 60)
    icon = SENTIMENT_EMOJI.get(parsed["sentiment"], "❓")
    print(f"{icon} {parsed['sentiment']}")

In [7]:
text = input("Paste the text to summarize: ")

raw_response = get_structured_summary(text)
parsed = parse_summary(raw_response)
print_summary(parsed)

SUMMARY
------------------------------------------------------------
Solar cars are electric vehicles powered by photovoltaic (PV) cells that convert sunlight into electricity, often supplemented by rechargeable batteries and regenerative braking. They draw on technology from aerospace, bicycle, alternative energy, and automotive industries, with design heavily focused on energy efficiency. While most solar cars have been built for racing competitions such as the World Solar Challenge, efforts to produce road-legal solar vehicles for consumers have faced significant commercial challenges, including the bankruptcy of Lightyear and production delays for the Aptera. The history of solar cars dates back to 1955 with General Motors' Sunmobile, and the technology has since advanced to solar arrays capable of producing over 2 kilowatts of power.

KEY POINTS
------------------------------------------------------------
1. Solar cars use photovoltaic cells to convert sunlight directly into elect